# ADSP Rare Variant Burden Analysis Pipeline

Complete workflow for gene-based rare variant burden testing using REGENIE.

**Pipeline Steps:**
1. Load ADSP dataset
2. Define cases and controls
3. Prepare phenotype files
4. Extract and filter genotypes by MAF
5. Create gene annotations
6. Run REGENIE Step 1 (null model)
7. Run REGENIE Step 2 (burden tests)

**Author:** Marzieh Khani 

**Date:** 03/09/2026


## Generate PCA using Genotools

In [ ]:
%%writefile ${WORK_DIR}/genotools_pc_all_ancestries.slurm
#!/bin/bash
#SBATCH --job-name=genotools_pca
#SBATCH --output=%x_%A_%a.out
#SBATCH --error=%x_%A_%a.err
#SBATCH --mem=100G
#SBATCH --cpus-per-task=4
#SBATCH --time=120:00:00
#SBATCH --array=0-10

# list of ancestries
ancestries=(EUR AFR AMR AAC AJ MDE FIN CAH CAS SAS EAS)

# Safety check: make sure SLURM_ARRAY_TASK_ID is valid
if [ $SLURM_ARRAY_TASK_ID -ge ${#ancestries[@]} ]; then
  echo "Error: Task ID $SLURM_ARRAY_TASK_ID exceeds number of ancestries"
  exit 1
fi

# Initialize conda
eval "$(${WORK_DIR}/miniconda3/bin/conda shell.bash hook)"
export PATH="${WORK_DIR}/miniconda3/bin:$PATH"
conda activate py311_genotools

# Clear Numba cache to avoid stale file handle error
NUMBA_CACHE_DIR="/tmp/numba_cache_$SLURM_ARRAY_TASK_ID"
rm -rf "$NUMBA_CACHE_DIR"
mkdir -p "$NUMBA_CACHE_DIR"
export NUMBA_CACHE_DIR

# Get the current ancestry for this job
anc="${ancestries[$SLURM_ARRAY_TASK_ID]}"

# Define input and output
WORK_DIR="${WORK_DIR}"
INPUT_PFILE="${INPUT_DIR}/output_${anc}"
OUTPUT_PREFIX="$WORK_DIR/PCA_output_${anc}"

echo "Processing ancestry: $anc (Task $SLURM_ARRAY_TASK_ID)"
echo "Input: $INPUT_PFILE"
echo "Output: $OUTPUT_PREFIX"

# Run GenoTools PCA
genotools \
  --pfile "$INPUT_PFILE" \
  --pca 10 \
  --out "$OUTPUT_PREFIX"

echo "Done: $anc"

In [ ]:
!sbatch genotools_pc_all_ancestries.slurm

## Load phenotype data

In [ ]:
import pandas as pd
import numpy as np

ancestry = 'EUR'  # Change this for different ancestries
base_dir = '${WORK_DIR}'
raw_pheno_file = '${ADSP_DIR}/ADSPIntegratedPhenotypes_DS_20250620.csv'

print("Step 1: Loading PC data...")
pcs = pd.read_csv(f'{base_dir}/PCA_output_{ancestry}.eigenvec', sep='\s+')    
pcs = pcs.rename(columns={'#FID': 'FID'})
print(f"  Loaded {len(pcs)} samples with PCs")

print("\nStep 2: Loading raw phenotype data...")
adsp_pheno = pd.read_csv(raw_pheno_file)
print(f"  Loaded {len(adsp_pheno)} total samples from ADSP")

# Keep only necessary columns
adsp_columns_to_keep = ['SampleID', 'Age_harmonized', 'Sex', 'DX_harmonized', 'other_diagnosis_flag']
adsp_pheno = adsp_pheno[adsp_columns_to_keep]

In [ ]:
adsp_pheno['DX_harmonized'].value_counts()

## Creating phenotype column

In [ ]:
# Assign phenotypes
# Controls: DX=0 AND other_diagnosis_flag != 1 -> PHENO = 1
# Cases: DX=1 -> PHENO = 2
# Missing: everything else -> PHENO = -9

def assign_pheno(row):
    # Check if DX_harmonized is missing
    dx_missing = pd.isna(row['DX_harmonized'])
    
    # If DX is missing, return -9
    if dx_missing:
        return -9 
    
    # Cases: has disease (DX=1) - assign PHENO=2 regardless of other_diagnosis_flag
    if row['DX_harmonized'] == 1:
        return 2
    
    # Controls: no disease (DX=0)
    elif row['DX_harmonized'] == 0:
        # If other_diagnosis_flag is missing, assume clean control
        other_dx_missing = pd.isna(row['other_diagnosis_flag'])
        if other_dx_missing:
            return 1
        
        # Clean control: no disease AND no other diagnosis
        elif row['other_diagnosis_flag'] != 1:
            return 1
        else:
            return -9
    
    # Everything else
    else:
        return -9
        
adsp_pheno['DX_harmonized'] = pd.to_numeric(adsp_pheno['DX_harmonized'], errors='coerce')
adsp_pheno['other_diagnosis_flag'] = pd.to_numeric(adsp_pheno['other_diagnosis_flag'], errors='coerce')

adsp_pheno['PHENO'] = adsp_pheno.apply(assign_pheno, axis=1)

# Create metadata dataframe
meta = pd.DataFrame({
    'FID': adsp_pheno['SampleID'],
    'IID': adsp_pheno['SampleID'],
    'PHENO': adsp_pheno['PHENO'],
    'AGE': adsp_pheno['Age_harmonized'],
    'SEX': adsp_pheno['Sex']
})

# Clean AGE column
meta['AGE'] = meta['AGE'].replace('90+', 90)
meta['AGE'] = pd.to_numeric(meta['AGE'], errors='coerce')

# Clean SEX column
meta['SEX'] = pd.to_numeric(meta['SEX'], errors='coerce')

print(f"  Created metadata with {len(meta)} samples")
print(f"  Phenotype distribution:")
print(meta['PHENO'].value_counts().sort_index())

merged = meta.merge(pcs, on=['FID', 'IID'], how='inner')
print(f"  Merged dataset has {len(merged)} samples")

filled_missing = merged.replace('', -9).fillna(-9)
filled_missing = filled_missing.replace(['NA', 'N/A', 'na', 'n/a', 'NULL', 'null'], -9)

## Create Phenotype and Covariates Files

In [ ]:
# Extract only FID, IID, PHENO for phenotype file
pheno_only = filled_missing[['FID', 'IID', 'PHENO']].copy()
pheno_file_out = f'{base_dir}/pheno_{ancestry}.txt'
pheno_only.to_csv(pheno_file_out, sep='\t', index=False)
print(f"  Saved: {pheno_file_out}")
print(f"  Samples in phenotype file: {len(pheno_only)}")

# Copy merged table for covariates
covar_file = filled_missing.copy()

# Drop PHENO column but KEEP SEX
if 'PHENO' in covar_file.columns:
    covar_file = covar_file.drop('PHENO', axis=1)

covar_file_out = f'{base_dir}/covariates_{ancestry}.txt'
covar_file.to_csv(covar_file_out, sep='\t', index=False)
print(f"  Saved: {covar_file_out}")
print(f"  Samples in covariates file: {len(covar_file)}")

# === FINAL SUMMARY ===
print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
print(f"Total samples: {len(filled_missing)}")

print(f"\nPhenotype distribution:")
pheno_counts = filled_missing['PHENO'].value_counts().sort_index()
for pheno, count in pheno_counts.items():
    pct = count / len(filled_missing) * 100
    print(f"  PHENO={pheno}: {count} samples ({pct:.1f}%)")

print(f"\nCovariates included:")
covar_cols = [col for col in covar_file.columns if col not in ['FID','IID']]
for col in covar_cols:
    print(f"  - {col}")


In [ ]:
import pandas as pd
import glob

base_dir = "${WORK_DIR}"

# Find all covariate files
files = glob.glob(f"{base_dir}/covariates_*.txt")

print("Found files:")
for f in files:
    print("  -", f)

# Load and merge
df_list = []
for f in files:
    temp = pd.read_csv(f, sep="\t")
    df_list.append(temp)

# Concatenate into one dataframe
merged = pd.concat(df_list, axis=0, ignore_index=True)

print("\nMerged dataset shape:", merged.shape)

# Save
out_path = f"{base_dir}/covariates_All_ancestries.txt"
merged.to_csv(out_path, sep="\t", index=False)

print("\nSaved:", out_path)


## Add pheno to the original files and clean the data

In [ ]:
%%bash
module load plink/2.0

# List of all ancestries
ancestries="AAC AFR AJ AMR CAH CAS EAS FIN MDE SAS EUR"

for anc in $ancestries
do
    echo "Running PLINK2 for ancestry: $anc"

    plink2 \
      --pfile ${INPUT_DIR}/output_${anc} \
      --pheno ${WORK_DIR}/pheno_${anc}.txt \
      --pheno-name PHENO \
      --make-bed \
      --require-pheno \
      --allow-extra-chr \
      --out output_ADSP_${anc}_with_pheno
done

echo "All ancestries completed!"


In [ ]:
%%bash
ancestries="AAC AFR AJ AMR CAH CAS EAS FIN MDE SAS EUR"

for anc in $ancestries
do
    echo "=== $anc ==="
    
    # Extract the phenotype column (6th column in .fam) and count values
    awk '{counts[$6]++} END {for (v in counts) print v, counts[v]}' \
        output_ADSP_${anc}_with_pheno.fam | sort -k1,1
    
    echo ""
done


In [ ]:
import pandas as pd

# Base directory
base_dir = "${WORK_DIR}"

# List of all ancestries
ancestries = ["AAC","AFR","AJ","AMR","CAH","CAS","EAS","MDE","SAS","EUR"]

for ancestry in ancestries:
    pheno_file = f"{base_dir}/pheno_{ancestry}.txt"
    clean_pheno_file = f"{base_dir}/pheno_{ancestry}_onlycasecontrol.txt"
    
    # Load phenotype file
    pheno = pd.read_csv(pheno_file, sep="\t")
    
    # Keep only valid case/control samples (PHENO = 1 or 2)
    pheno_clean = pheno[pheno['PHENO'].isin([1, 2])]
    
    # Save cleaned phenotype file
    pheno_clean.to_csv(clean_pheno_file, sep="\t", index=False)
    
    # Print summary
    print(f"{ancestry}:")
    print(f"  Original samples: {len(pheno)}")
    print(f"  Kept samples: {len(pheno_clean)}")
    print(f"  Phenotype counts:\n{pheno_clean['PHENO'].value_counts()}")
    print("-"*50)


In [ ]:
import pandas as pd

# Base directory
base_dir = "${WORK_DIR}"

# List of ancestries
ancestries = ["AAC","AFR","AJ","AMR","CAH","CAS","EAS","MDE","SAS","EUR"]

for anc in ancestries:
    # Load cleaned phenotype file
    pheno_file = f"{base_dir}/pheno_{anc}_onlycasecontrol.txt"
    pheno = pd.read_csv(pheno_file, sep="\t")
    
    # Load covariates file
    cov_file = f"{base_dir}/covariates_{anc}.txt"
    cov = pd.read_csv(cov_file, sep="\t")
    
    # Keep only rows where FID/IID are in the cleaned pheno file
    cov_clean = cov[cov['FID'].isin(pheno['FID']) & cov['IID'].isin(pheno['IID'])]
    
    # Save cleaned covariate file
    cov_clean_file = f"{base_dir}/covariates_{anc}_onlycasecontrol.txt"
    cov_clean.to_csv(cov_clean_file, sep="\t", index=False)
    
    # Print summary
    print(f"{anc}:")
    print(f"  Original covariates samples: {len(cov)}")
    print(f"  Kept samples after cleaning: {len(cov_clean)}")
    print("-"*50)


In [ ]:
import pandas as pd

# Base directory
base_dir = "${WORK_DIR}"

# List of ancestries
ancestries = ["AAC","AFR","AJ","AMR","CAH","CAS","EAS","MDE","SAS","EUR"]

# List to store dataframes
cov_list = []

for anc in ancestries:
    cov_file = f"{base_dir}/covariates_{anc}_onlycasecontrol.txt"
    cov = pd.read_csv(cov_file, sep="\t")
    cov_list.append(cov)
    print(f"{anc}: {len(cov)} samples added")

# Concatenate all dataframes
cov_all = pd.concat(cov_list, ignore_index=True)

# Save merged covariate file
output_file = f"{base_dir}/covariates_All_ancestries_onlycasecontrol.txt"
cov_all.to_csv(output_file, sep="\t", index=False)

print(f"\nMerged covariates saved to: {output_file}")
print(f"Total samples: {len(cov_all)}")


## QC and MAF stratification 

In [ ]:
%%writefile run_qc_all.slurm
#!/bin/bash
#SBATCH --job-name=ADSP_QC
#SBATCH --output=ADSP_QC_%A_%a.out
#SBATCH --error=ADSP_QC_%A_%a.err
#SBATCH --mem=400G
#SBATCH --cpus-per-task=4
#SBATCH --time=24:00:00

module load plink/2.0

ancestries="AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR"

for anc in $ancestries
do
    echo "Processing ancestry: $anc"
    infile="output_ADSP_${anc}_with_pheno"
    PFX="${anc}_raw"

    plink2 --bfile $infile --make-pgen --out $PFX

    awk 'NR>1 {key=$1":"$2; c[key]++} END{for(k in c) if(c[k]>1) print k}' ${PFX}.pvar > dup.pos

    awk 'BEGIN{FS=OFS="\t"}
         NR==FNR {dup[$1]=1; next}
         NR==1 {next}
         {
           key=$1":"$2
           is_snp=(length($4)==1 && length($5)==1 && $4~/^[ACGT]$/ && $5~/^[ACGT]$/)
           if(is_snp && !(key in dup)) print $3
         }' dup.pos ${PFX}.pvar > keep.ids

    plink2 --pfile $PFX \
           --extract keep.ids \
           --geno 0.05 \
           --hwe 1e-6 midp \
           --make-pgen \
           --out ${anc}_clean

    for maf in 0.01 0.03 0.05
    do
        plink2 --pfile ${anc}_clean \
               --max-maf $maf \
               --mac 1 \
               --make-pgen \
               --out ${anc}_MAF$(echo $maf | sed 's/0\.//')
    done

    echo " Ancestry $anc completed"
done


In [ ]:
!sbatch run_qc_all.slurm

## Extracting coding variants

In [ ]:
%%writefile extract_coding_array.slurm
#!/bin/bash
#SBATCH --job-name=extract_coding
#SBATCH --output=extract_coding_%A_%a.out
#SBATCH --error=extract_coding_%A_%a.err
#SBATCH --cpus-per-task=8
#SBATCH --mem=150G
#SBATCH --time=12:00:00
#SBATCH --array=0-8

set -euo pipefail

module load plink/2.0
module load bedtools

# Ancestry list 
ancestries=(AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR)

anc=${ancestries[$SLURM_ARRAY_TASK_ID]}

maf_codes=("01" "03" "05")


# Step 1: Create CDS BED file 

if [ ! -f gencode.v38.annotation.gtf.gz ]; then
    echo "Downloading GENCODE v38 annotation..."
    wget -q https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38/gencode.v38.annotation.gtf.gz
fi

echo "Creating CDS BED file..."
zcat gencode.v38.annotation.gtf.gz \
    | awk '$3=="CDS"' \
    | awk -v OFS="\t" '{print $1, $4-1, $5}' \
    | sort -k1,1 -k2,2n \
    | bedtools merge -i - \
    > cds_hg38.bed

echo " CDS BED created with $(wc -l < cds_hg38.bed) regions"
echo ""

# Step 2: Process all MAF thresholds for this ancestry

for maf in "${maf_codes[@]}"; do
    infile="${anc}_MAF${maf}"

    echo "---- MAF ${maf} ----"

    if [ ! -f "${infile}.pgen" ]; then
        echo " Missing file: ${infile}.pgen"
        continue
    fi

    # Extract coding variants
    plink2 \
        --pfile ${infile} \
        --extract bed1 cds_hg38.bed \
        --make-pgen \
        --out ${infile}_coding

    # Variant counts
    total=$(tail -n+2 ${infile}.pvar | wc -l)
    coding=$(tail -n+2 ${infile}_coding.pvar | wc -l)
    pct=$(echo "scale=2; $coding*100/$total" | bc)

    echo " Coding variants for ${anc} MAF${maf}: $coding / $total (${pct}%)"
done



In [ ]:
!sbatch extract_coding_array.slurm

## Creating gene annotation files for REGENIE

In [ ]:
%%writefile create_annotations_array.slurm
#!/bin/bash
#SBATCH --job-name=anno
#SBATCH --output=anno_%A_%a.out
#SBATCH --error=anno_%A_%a.err
#SBATCH --cpus-per-task=4
#SBATCH --mem=80G
#SBATCH --time=8:00:00
#SBATCH --array=0-10

set -euo pipefail

module load bedtools
module load python

# Ancestry list 

ancestries=(AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR)
anc=${ancestries[$SLURM_ARRAY_TASK_ID]}

mafs=(01 03 05)


# Step 1: Get GENCODE + genes

if [ ! -f gencode.v38.annotation.gtf.gz ]; then
    echo "Downloading GENCODE v38..."
    wget -q https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38/gencode.v38.annotation.gtf.gz
fi

echo "Extracting protein-coding genes..."
zcat gencode.v38.annotation.gtf.gz | \
  awk '$3=="gene"' | \
  awk -v OFS='\t' '{
    match($0, /gene_name "([^"]+)"/, gname)
    match($0, /gene_type "([^"]+)"/, gtype)
    if(gtype[1]=="protein_coding") {
      chr=$1; gsub("chr","",chr)
      print chr, $4-1, $5, gname[1]
    }
  }' > genes_hg38.bed

echo " $(wc -l < genes_hg38.bed) genes"

# Step 2: Loop over MAF thresholds and vtypes

for maf in "${mafs[@]}"; do
    for vtype in all coding; do

        if [ "$vtype" == "all" ]; then
            pfx="${anc}_MAF${maf}"
        else
            pfx="${anc}_MAF${maf}_coding"
        fi

        echo ""
        echo "----------- ${pfx} -----------"

        if [ ! -f "${pfx}.pvar" ]; then
            echo " File missing: ${pfx}.pvar - skipping"
            continue
        fi

        # Make variant BED file
        awk 'NR>1 {print $1"\t"($2-1)"\t"$2"\t"$3"\t"$4"\t"$5}' \
            ${pfx}.pvar > ${pfx}_variants.bed

        echo "Mapping variants to genes..."
        bedtools intersect \
            -a ${pfx}_variants.bed \
            -b genes_hg38.bed \
            -wa -wb \
            | awk '{print $1"\t"($2+1)"\t"$4"\t"$5"\t"$9}' \
            > ${pfx}_anno.txt

        # Gene list
        cut -f5 ${pfx}_anno.txt | sort -u > ${pfx}_sets.txt

        # Masks
        cat > ${pfx}_mask.txt << 'EOF'
# REGENIE mask definition file
# Format: MASK_NAME CONSEQUENCE_TYPES
LOF stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant
MISSENSE missense_variant
DAMAGING stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant,missense_variant
EOF

        echo " $(wc -l < ${pfx}_anno.txt) annotations created"
        echo " $(wc -l < ${pfx}_sets.txt) genes sets"
    done
done


In [ ]:
!sbatch create_annotations_array.slurm

## Extract Common Variants for REGENIE Step 1

In [ ]:
%%writefile extract_common_all_ancestries.slurm
#!/bin/bash
#SBATCH --job-name=extract_common_step1
#SBATCH --output=extract_common_%A_%a.out
#SBATCH --error=extract_common_%A_%a.err
#SBATCH --cpus-per-task=4
#SBATCH --mem=150G
#SBATCH --time=12:00:00

module load plink/2.0

ancestries="AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR"


for anc in $ancestries; do

    # 1) Extract COMMON variants
    plink2 --pfile ${anc}_raw \
           --maf 0.05 \
           --mac 20 \
           --geno 0.05 \
           --hwe 1e-6 \
           --make-pgen \
           --out ${anc}_common

    echo "   Extracted common variants"

    # 2) LD pruning
    plink2 --pfile ${anc}_common \
           --indep-pairwise 1000 100 0.5 \
           --out ${anc}_common_pruned

    # safety check
    if [ ! -f ${anc}_common_pruned.prune.in ]; then
        echo "   ERROR: No prune.in file created for $anc"
        exit 1
    fi

    prune_count=$(wc -l < ${anc}_common_pruned.prune.in)
    echo "   LD-pruned variants: $prune_count"

    # 3) Extract pruned variants
    plink2 --pfile ${anc}_common \
           --extract ${anc}_common_pruned.prune.in \
           --make-pgen \
           --out ${anc}_step1_variants

    final_count=$(tail -n+2 ${anc}_step1_variants.pvar | wc -l)
    echo "   Final Step 1 variants for REGENIE: $final_count"

    echo "Output files:"
    echo "  ${anc}_step1_variants.pgen"
    echo "  ${anc}_step1_variants.pvar"
    echo "  ${anc}_step1_variants.psam"
    echo ""
done


In [ ]:
!sbatch extract_common_all_ancestries.slurm

## Filter controls with AGE ≥ 65 and regenerate genotype and phenotype data

In [ ]:
import pandas as pd

# List of ancestries
ancestries = ["AAC","AFR","AJ","AMR","CAH","CAS","EAS","MDE","SAS","EUR"]

for anc in ancestries:
    print(f"\nProcessing ancestry: {anc}")
    
    # Load phenotype and covariates
    pheno_file = f"pheno_{anc}_onlycasecontrol.txt"
    cov_file = f"covariates_{anc}_onlycasecontrol.txt"
    
    pheno = pd.read_csv(pheno_file, sep="\t")
    cov = pd.read_csv(cov_file, sep="\t")
    
    # Merge phenotype and covariates on FID and IID
    df = pheno.merge(cov, on=["FID","IID"])
    
    # Count before filtering
    n_cases_before = (df['PHENO'] == 2).sum()
    n_controls_before = (df['PHENO'] == 1).sum()
    
    print(f"Before filtering: Cases = {n_cases_before}, Controls = {n_controls_before}")
    
    # Filter: keep cases + controls with AGE >=65
    df_filtered = df[(df['PHENO'] == 2) | ((df['PHENO'] == 1) & (df['AGE'] >= 65))]
    
    # Count after filtering
    n_cases_after = (df_filtered['PHENO'] == 2).sum()
    n_controls_after = (df_filtered['PHENO'] == 1).sum()
    
    print(f"After filtering: Cases = {n_cases_after}, Controls = {n_controls_after}")
    
    # Save new phenotype file
    pheno_filtered = df_filtered[['FID','IID','PHENO']]
    pheno_filtered.to_csv(f"pheno_{anc}_onlycasecontrol_age65.txt", sep="\t", index=False)
    
    # Save new covariates file
    cov_filtered = df_filtered[['FID','IID','AGE','SEX','PC1','PC2','PC3','PC4','PC5',
                                'PC6','PC7','PC8','PC9','PC10']]
    cov_filtered.to_csv(f"covariates_{anc}_onlycasecontrol_age65.txt", sep="\t", index=False)
    
    print(f"Filtered files saved for {anc}.")


In [ ]:
import pandas as pd

ancestries = ["AAC","AFR","AJ","AMR","CAH","CAS","EAS","MDE","SAS","EUR"]

all_cov = []

for anc in ancestries:
    cov_file = f"covariates_{anc}_onlycasecontrol_age65.txt"
    df = pd.read_csv(cov_file, sep="\t")
    all_cov.append(df)

# Concatenate all ancestries
df_all = pd.concat(all_cov, ignore_index=True)

# Save merged file
df_all.to_csv("covariates_All_ancestries_onlycasecontrol_age65.txt", sep="\t", index=False)

print(f" Merged covariates saved: covariates_All_ancestries_onlycasecontrol_age65.txt")
print(f"Total samples: {len(df_all)}")


In [ ]:
%%bash
#!/bin/bash

ancestries="AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR"

module load plink/2.0

for anc in $ancestries
do
    echo "Processing $anc"

    plink2 \
      --pfile ${anc}_step1_variants \
      --keep pheno_${anc}_onlycasecontrol_age65.txt \
      --make-pgen \
      --out ${anc}_step1_variants_age65

    echo " Done: ${anc}"
done


In [ ]:
%%bash
#!/bin/bash

module load plink/2.0

ancestries="AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR"
mafs="01 03 05"   

for anc in $ancestries
do

    keepfile="pheno_${anc}_onlycasecontrol_age65.txt"

    for maf in $mafs
    do

        # 1) Full rare variant set
        infile="${anc}_MAF${maf}"

        if [[ -f ${infile}.pgen ]]; then
            echo "    Filtering ${infile} ..."
            plink2 \
              --pfile ${infile} \
              --keep ${keepfile} \
              --make-pgen \
              --out ${infile}_age65
        else
            echo "    SKIPPED: ${infile}.pgen not found."
        fi

        # 2) Coding-only rare variants
        infile_coding="${anc}_MAF${maf}_coding"

        if [[ -f ${infile_coding}.pgen ]]; then
            echo "    Filtering ${infile_coding} ..."
            plink2 \
              --pfile ${infile_coding} \
              --keep ${keepfile} \
              --make-pgen \
              --out ${infile_coding}_age65
        else
            echo "    SKIPPED: ${infile_coding}.pgen not found."
        fi
    done

    echo " Done with $anc."
    echo ""
done


## Recodes sex from 0/1 to 1/2

In [ ]:
%%bash
#!/bin/bash

ancestries="AAC AFR AJ AMR CAH CAS EAS EUR MDE SAS"

for anc in $ancestries; do
    infile="covariates_${anc}_onlycasecontrol_age65.txt"
    outfile="covariates_${anc}_onlycasecontrol_age65_sex12.txt"

    if [[ ! -f "$infile" ]]; then
        echo " $infile NOT FOUND"
        continue
    fi

    echo "Processing $infile -> $outfile"

    awk '
        BEGIN { FS=OFS="\t" }
        
        NR==1 {
            # find the SEX column index
            for (i=1; i<=NF; i++) {
                if ($i == "SEX") sexcol = i
            }
            print
            next
        }

        {
            if ($sexcol == 0) $sexcol = 1    # male
            else if ($sexcol == 1) $sexcol = 2  # female
            print
        }
    ' "$infile" > "$outfile"

    echo " Saved $outfile"
    echo ""
done



In [ ]:
%%bash
#!/bin/bash

# List your ancestries
ancestries="AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR"

outfile="covariates_All_ancestries_onlycasecontrol_age65_sex12.txt"

# Start fresh
rm -f $outfile

header_written=false

for anc in $ancestries; do
    infile="covariates_${anc}_onlycasecontrol_age65_sex12.txt"

    if [[ ! -f "$infile" ]]; then
        echo " Missing: $infile"
        continue
    fi

    echo "Adding $infile ..."

    # Write header from first file only
    if ! $header_written; then
        head -n 1 "$infile" > "$outfile"
        header_written=true
    fi

    # Append data rows
    tail -n +2 "$infile" >> "$outfile"
done



## Selected data for chromosomes 1–22

In [ ]:
import subprocess

ancestries = ["AAC","AFR","AJ","AMR","CAH","CAS","EAS","MDE","SAS","EUR"]

for anc in ancestries:
    print(f"Filtering {anc} to autosomes...")
    cmd = f"""
    module load plink/2.0
    plink2 --pfile {anc}_step1_variants_age65 \
           --chr 1-22 \
           --make-pgen \
           --out {anc}_step1_variants_age65.autosomes
    """
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")



In [ ]:
import subprocess

# All ancestries
ancestries = ["AAC","AFR","AJ","AMR","CAH","CAS","EAS","MDE","SAS","EUR"]

# MAF groups
maf_groups = ["MAF01", "MAF03", "MAF05"]

# File types
file_types = ["age65", "coding_age65"]  

# Function to run bash commands with module load
def run(cmd):
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")

# Loop over ancestries, MAF, and file types
for anc in ancestries:
    for maf in maf_groups:
        for ftype in file_types:
            input_prefix = f"{anc}_{maf}_{ftype}"
            output_prefix = f"{input_prefix}.autosomes"
            print(f"Filtering {input_prefix} to autosomes...")

            cmd = f"""
            module load plink/2.0
            plink2 --pfile {input_prefix} \
                   --chr 1-22 \
                   --make-pgen \
                   --out {output_prefix}
            """
            run(cmd)


## Install REGENIE

In [ ]:
%%bash

# 1. Create bin and software directories

mkdir -p $HOME/bin
mkdir -p $HOME/software
cd $HOME/software


# 2. Remove old downloaded files if they exist

rm -f regenie_v3.2.9.gz_x86_64_Linux
rm -f regenie_v3.2.9.gz_x86_64_Linux.zip


# 3. Download latest regenie v3.2.9 binary

wget https://github.com/rgcgithub/regenie/releases/download/v3.2.9/regenie_v3.2.9.gz_x86_64_Linux.zip


# 4. Unzip (overwrite if needed)

unzip -o regenie_v3.2.9.gz_x86_64_Linux.zip


# 5. Move executable to ~/bin and rename to regenie

rm -f $HOME/bin/regenie     # remove old regenie if exists
mv regenie_v3.2.9.gz_x86_64_Linux $HOME/bin/regenie


# 6. Give execute permission

chmod +x $HOME/bin/regenie

# 7. Add ~/bin to PATH if not already added

if ! grep -q 'export PATH=$HOME/bin:$PATH' ~/.bashrc; then
    echo 'export PATH=$HOME/bin:$PATH' >> ~/.bashrc
fi

# Reload PATH immediately
source ~/.bashrc

# 8. Test installation

regenie --help


## REGENIE Step 1 - Fit Null Model

In [ ]:
#!/usr/bin/env python3

ancestries = ["AAC", "AFR", "AJ", "AMR", "CAH", "CAS", "EAS", "MDE", "SAS", "EUR"]

for anc in ancestries:

    PGEN = f"{anc}_step1_variants_age65.autosomes"
    PHENO = f"pheno_{anc}_onlycasecontrol_age65.txt"
    COVAR = f"covariates_{anc}_onlycasecontrol_age65_sex12.txt"

    script = f"""#!/bin/bash
#SBATCH --job-name=regenie_step1_{anc}
#SBATCH --cpus-per-task=48
#SBATCH --mem=180g
#SBATCH --time=24:00:00
#SBATCH --partition=norm
#SBATCH --output=step1_{anc}.out
#SBATCH --error=step1_{anc}.err

set -euo pipefail

# Use manually installed regenie
export PATH=$HOME/bin:$PATH

echo " REGENIE Step 1 - {anc}"

echo "Checking files..."
ls -lh {PGEN}.pgen {PGEN}.pvar {PGEN}.psam
ls -lh {PHENO}
ls -lh {COVAR}

VAR_COUNT=$(tail -n +2 {PGEN}.pvar | wc -l)
echo "Variant count: $VAR_COUNT"

if [ "$VAR_COUNT" -lt 1000 ]; then
    echo "ERROR: Too few variants (<1000)"
    exit 1
fi

regenie \\
  --step 1 \\
  --pgen {PGEN} \\
  --phenoFile {PHENO} \\
  --covarFile {COVAR} \\
  --phenoCol PHENO \\
  --covarColList SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \\
  --bt \\
  --cc12 \\
  --bsize 1000 \\
  --threads 48 \\
  --lowmem \\
  --lowmem-prefix tmp_{anc} \\
  --force-step1 \\
  --gz \\
  --out step1_{anc}

echo "Checking LOCO files..."
ls step1_{anc}_*.loco.gz | wc -l || true
echo "Done."
"""

    filename = f"regenie_step1_{anc}.sh"
    with open(filename, "w") as f:
        f.write(script)

    print(f"Created: {filename}")



In [ ]:
%%bash
for anc in AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR; do
    sbatch regenie_step1_${anc}.sh
done


## Fix Annotation Files for REGENIE

In [ ]:
%%writefile create_annotations_step2_final_array.slurm
#!/bin/bash
#SBATCH --job-name=anno_step2
#SBATCH --output=anno_step2_%A_%a.out
#SBATCH --error=anno_step2_%A_%a.err
#SBATCH --cpus-per-task=4
#SBATCH --mem=80G
#SBATCH --time=8:00:00
#SBATCH --array=0-9

set -euo pipefail

module load bedtools
module load python

# Ancestry list 

ancestries=(AAC AFR AJ AMR CAH CAS EAS MDE SAS EUR)
anc=${ancestries[$SLURM_ARRAY_TASK_ID]}

# MAF thresholds
mafs=(01 03 05)

echo "  Processing ancestry: $anc"

# Step 2: Loop over MAF thresholds and vtypes

for maf in "${mafs[@]}"; do
    for vtype in all coding; do

        # Define filenames
        if [ "$vtype" == "all" ]; then
            pfx="${anc}_MAF${maf}_age65.autosomes"
        else
            pfx="${anc}_MAF${maf}_coding_age65.autosomes"
        fi

        echo ""
        echo "----------- ${pfx} -----------"

        # Check .pvar exists
        if [ ! -f "${pfx}.pvar" ]; then
            echo " File missing: ${pfx}.pvar - skipping"
            continue
        fi

        # Variant BED file for bedtools
        
        awk 'NR>1 {print $1"\t"($2-1)"\t"$2"\t"$3"\t"$4"\t"$5}' \
            ${pfx}.pvar > ${pfx}_variants_gene.bed

       
        # Map variants to genes
        
        bedtools intersect \
            -a ${pfx}_variants_gene.bed \
            -b genes_hg38.bed \
            -wa -wb \
        | awk '{
            chr=$1;
            pos=$3;      
            ref=$4;
            alt=$5;
            gene=$10;   
            print chr"\t"pos"\t"ref"\t"alt"\t"gene
        }' \
        > ${pfx}_anno_gene.txt

        # Create gene set file
        
        cut -f5 ${pfx}_anno_gene.txt | sort -u > ${pfx}_sets_gene.txt

        
        # Create mask file 
        
        cat > ${pfx}_mask_gene.txt << 'EOF'
# REGENIE mask definition file
# Format: MASK_NAME CONSEQUENCE_TYPES
LOF stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant
MISSENSE missense_variant
DAMAGING stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant,missense_variant
EOF

        echo " $(wc -l < ${pfx}_anno_gene.txt) annotations created"
        echo " $(wc -l < ${pfx}_sets_gene.txt) gene sets created"
        echo " Mask file ${pfx}_mask_gene.txt created"

    done
done



In [ ]:
! sbatch create_annotations_step2_final_array.slurm

In [ ]:
#!/usr/bin/env python3

import os

ANCESTRIES = [
    "AAC", "AFR", "AJ", "AMR", "CAH",
    "CAS", "EAS", "EUR", "MDE", "SAS"
]

MAF_CODES = ["01", "03", "05"]
VTYPES = ["", "_coding"]   

print("="*80)
print("BUILDING ALL REGENIE ANNOTATION FILES")
print("="*80)

# Helper

def read_lines(path):
    with open(path, "r") as f:
        return [l.strip() for l in f if l.strip()]

# Main Loop

for anc in ANCESTRIES:
    print(f"\n\n### Processing ancestry: {anc}")
    for maf in MAF_CODES:
        for vt in VTYPES:

            vt_name = "all" if vt == "" else "coding"
            prefix = f"{anc}_MAF{maf}{vt}_age65.autosomes"

            anno_in = f"{prefix}_anno_gene.txt"
            sets_in = f"{prefix}_sets_gene.txt"
            mask_in = f"{prefix}_mask_gene.txt"

            # --- Check files exist ---
            if not os.path.exists(anno_in):
                print(f"   Skipping {prefix} (missing {anno_in})")
                continue

            print(f"\n  -> {prefix}")
            print("     Input:", anno_in)

            # Part 1: Build REGENIE-style annotation (variant . gene)
            
            anno_regenie = f"{prefix}_anno_regenie.txt"
            with open(anno_in, "r") as fin, open(anno_regenie, "w") as fout:
                for line in fin:
                    parts = line.strip().split()
                    var = parts[2]   
                    gene = parts[-1] 
                    fout.write(f"{var}\t.\t{gene}\n")

            print("      Built:", anno_regenie)

            
            # Part 2: Build M1 annotation (variant M1 gene)
           
            anno_m1 = f"{prefix}_anno_M1.txt"
            with open(anno_regenie, "r") as fin, open(anno_m1, "w") as fout:
                for line in fin:
                    parts = line.strip().split()
                    var, _, gene = parts
                    fout.write(f"{var}\tM1\t{gene}\n")

            print("      Built:", anno_m1)

            
            # Part 3: Convert to REGENIE Proper 4-column Format
            # Format: VARIANT  GENE  REGION  ANNOTATION
            
            anno_proper = f"{prefix}_anno_proper4.txt"
            with open(anno_m1, "r") as fin, open(anno_proper, "w") as fout:
                for line in fin:
                    var, _, gene = line.strip().split()
                    fout.write(f"{var}\t{gene}\t.\tALL\n")

            print("      Built:", anno_proper)

            
            # Part 4: Build set-list (GENE  CHR  START  VAR1,VAR2,...)
            
            setdict = {}   # gene -> {chr, start, list}
            with open(anno_m1, "r") as fin:
                for line in fin:
                    var, _, gene = line.strip().split()

                    # Parse variant ID
                    chrom = var.split(":")[0]
                    pos = int(var.split(":")[1])

                    if gene not in setdict:
                        setdict[gene] = {
                            "chr": chrom,
                            "start": pos,
                            "vars": [var]
                        }
                    else:
                        setdict[gene]["vars"].append(var)
                        if pos < setdict[gene]["start"]:
                            setdict[gene]["start"] = pos

            sets_proper = f"{prefix}_sets_proper4.txt"
            with open(sets_proper, "w") as fout:
                for gene in sorted(setdict.keys()):
                    chrom = setdict[gene]["chr"]
                    start = setdict[gene]["start"]
                    varlist = ",".join(setdict[gene]["vars"])
                    fout.write(f"{gene}\t{chrom}\t{start}\t{varlist}\n")

            print("      Built:", sets_proper)

            
            # Part 5: Build Mask 
            
            mask_proper = f"{prefix}_mask_proper4.txt"
            with open(mask_proper, "w") as fout:
                fout.write("M1\tALL\n")

            print("      Built:", mask_proper)



## REGENIE Step 2

In [ ]:
#!/usr/bin/env python3


import os
import subprocess

# ---------- CONFIG ----------
ANCESTRIES = ["AAC","AFR","AJ","AMR","CAH","CAS","EAS","EUR","MDE","SAS"]
MAF_CODES = ["01","03","05"]  
VAR_TYPES = ["all","coding"]
THREADS = 96
MEM = "180g"
TIME = "96:00:00"
SUBMIT_JOBS = True 



for anc in ANCESTRIES:
    pheno_file = f"pheno_{anc}_onlycasecontrol_age65.txt"
    covar_file = f"covariates_{anc}_onlycasecontrol_age65_sex12.txt"
    pred_file = f"step1_{anc}_pred.list"

    for maf in MAF_CODES:
        maf_label = f"MAF{maf}"  

        for vtype in VAR_TYPES:

            suffix = "" if vtype == "all" else "_coding"
            pfx = f"{anc}_{maf_label}{suffix}_age65.autosomes"

            scriptname = f"step2_{anc}_{maf_label}_{vtype}.sh"

            # sbatch script
            script = f"""#!/bin/bash
#SBATCH --job-name=step2_{anc}_{maf_label}_{vtype}
#SBATCH --cpus-per-task={THREADS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output=log_step2_{anc}_{maf_label}_{vtype}.out
#SBATCH --error=log_step2_{anc}_{maf_label}_{vtype}.err

set -euxo pipefail

# Use manually installed regenie
export PATH=$HOME/bin:$PATH

echo "Running REGENIE Step 2 for {anc}, {maf_label}, type {vtype}"

regenie \\
  --step 2 \\
  --pgen {pfx} \\
  --phenoFile {pheno_file} \\
  --covarFile {covar_file} \\
  --phenoCol PHENO \\
  --covarColList SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \\
  --bt \\
  --cc12 \\
  --pred {pred_file} \\
  --anno-file {pfx}_anno_proper4.txt \\
  --set-list {pfx}_sets_proper4.txt \\
  --mask-def {pfx}_mask_proper4.txt \\
  --vc-tests skato \\
  --threads {THREADS} \\
  --gz \\
  --out results_step2_{anc}_{maf_label}_{vtype}

echo "DONE: {anc} {maf_label} {vtype}"
"""

            with open(scriptname, "w") as f:
                f.write(script)

            print(f"Created sbatch script: {scriptname}")

            
            if SUBMIT_JOBS:
                result = subprocess.run(["sbatch", scriptname], capture_output=True, text=True)
                if result.returncode == 0:
                    job_id = result.stdout.strip()
                    print(f" Submitted job: {scriptname} (job ID: {job_id})")
                else:
                    print(f" Failed to submit {scriptname}:\n{result.stderr}")